In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Build an open lakehouse on Google Cloud

<table align="left">
  <td><a href="https://github.com/GoogleCloudPlatform/devrel-demos/blob/main/data-analytics/lakehouse/open_lakehouse_beam.ipynb"><img src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px" alt="GitHub logo"> View on GitHub</a></td>
  <td><a href="https://console.cloud.google.com/agent-platform/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/devrel-demos/blob/main/data-analytics/lakehouse/open_lakehouse_beam.ipynb"><img src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" alt="Vertex AI logo"> Open in Agent Platform Workbench</a></td>
</table>

In this notebook, you will create and connect to a [Lakehouse for Apache Iceberg](https://cloud.google.com/biglake) catalog using Apache Beam. You will:

- Create a Lakehouse catalog
- Load data from a public REST catalog
- Perform transformations on it using Apache Beam
- Write it to a new Catalog

## Setup



Set the log level to only show ERRORs

In [ ]:
import logging

# Set the root logger to ERROR level
logging.getLogger().setLevel(logging.ERROR)

Configure environment variables. Provide your project ID and a [region](https://cloud.google.com/bigquery/docs/locations#regions) to store your resources, such as `us-central1`. **Note**: this tutorial will not work with [multi-regions](https://cloud.google.com/bigquery/docs/locations#multi-regions).

In [ ]:
PROJECT_ID = "bmiro-external"
REGION = "us-central1"
WAREHOUSE_BUCKET_NAME = f"{PROJECT_ID}-test-warehouse"

Create a Cloud Storage bucket for your warehouse.

In [ ]:
from google.cloud import storage

storage_client = storage.Client()
if not storage_client.bucket(WAREHOUSE_BUCKET_NAME).exists():
    storage_client.create_bucket(WAREHOUSE_BUCKET_NAME, location=REGION)

## Create a Lakehouse Catalog

Create a [Lakehouse Catalog](https://docs.cloud.google.com/lakehouse/docs/lakehouse-iceberg-rest-catalog) that can be accessed from multiple query engines, including Beam.

In [ ]:
CATALOG_NAME = WAREHOUSE_BUCKET_NAME

import subprocess

command = [
    'gcloud',
    'alpha',
    'biglake',
    'iceberg',
    'catalogs',
    'create',
    CATALOG_NAME, # Use the bucket name as the name for the catalog
    '--catalog-type=gcs-bucket',
    '--credential-mode=vended-credentials',
    f'--primary-location={REGION}'
]

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Capture the service account used for this catalog.

In [ ]:
import re

match = re.search(r"BigLake service account: (.*)", result.stderr)

service_account = None
if match:
    service_account = match.group(1)

Give the catalog service account access to the warehouse bucket.

In [ ]:
command = [
    'gcloud',
    'storage',
    'buckets',
    'add-iam-policy-binding',
    f'gs://{WAREHOUSE_BUCKET_NAME}',
    f'--member',
    f'serviceAccount:{service_account}',
    '--role',
    'roles/storage.objectUser',
]

result = subprocess.run(command, capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

## Connect to the Lakehouse Catalog

Import necessary libaries.

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from statistics import mean

Set up the configuration to connect to the [public Lakehouse Runtime Catalog](https://opensource.googleblog.com/2026/01/explore-public-datasets-with-apache-iceberg-and-biglake.html). Note the following config:
- `type`: This denotes connecting to the catalog via a REST endpoint
- `uri`: Specifies the Lakehouse API
- `warehouse`: The Cloud Storage bucket where the Lakehouse lives
- `header.x-goog-user-project`: Your project ID 
- `rest.auth.type`: Use Google authentication
- `io-impl`: The dependency for reading Iceberg files to/from Cloud Storage
- `header-X-Iceberg-Access-Delegation`: Specify using [vendor credentials](https://docs.cloud.google.com/lakehouse/docs/lakehouse-iceberg-rest-catalog#create_a_catalog)

In [ ]:
# Set up the catalog properties
public_lakehouse_catalog_props = {
  'type': 'rest',
  'uri': 'https://biglake.googleapis.com/iceberg/v1/restcatalog',
  'warehouse': 'gs://biglake-public-nyc-taxi-iceberg',
  'header.x-goog-user-project': PROJECT_ID,
  'rest.auth.type': 'google',
  'io-impl': 'org.apache.iceberg.gcp.gcs.GCSFileIO',
  'header.X-Iceberg-Access-Delegation': 'vended-credentials'
}

# Set up the connection and use filtering to only load part of the data.
source_config = {
      "table": "public_data.nyc_taxicab",
      "catalog_properties": public_biglake_catalog_props,
      "filter": "data_file_year = 2021 AND tip_amount > 100",
      "keep": ["passenger_count", "total_amount", "trip_distance"]
}

Create similar configuration for your new catalog.

In [ ]:
new_lakehouse_catalog_props = {
  'type': 'rest',
  'uri': 'https://biglake.googleapis.com/iceberg/v1/restcatalog',
  'warehouse': f"gs://{WAREHOUSE_BUCKET_NAME}",
  'header.x-goog-user-project': PROJECT_ID,
  'rest.auth.type': 'google',
  'io-impl': 'org.apache.iceberg.gcp.gcs.GCSFileIO',
  'header.X-Iceberg-Access-Delegation': 'vended-credentials'
}

# The Iceberg connector will automatically create the namespace 'my_namespace' and table 'my_table'.
sink_config = {
      "table": "my_namespace.nyc_taxicab",
      "catalog_properties": new_lakehouse_catalog_props,
}

Run the job:
- First, the data is loaded from the public catalog.
- Second, 

In [ ]:
# Create a helper function to cast the loaded data to the proper data types.
def to_formal_row(element):
    return beam.Row(
        # Force these to standard types (int/float)
        passenger_count=int(element.passenger_count) if element.passenger_count is not None else 0,
        total_trips=int(element.total_trips),
        avg_fare=float(element.avg_fare),
        avg_distance=float(element.avg_distance)
    )


#   
with beam.Pipeline(options=options) as p:
    rows = p | "ReadFromIceberg" >> beam.managed.Read(
        "iceberg", config=source_config
    )

    result = (
    rows | beam.Select(num_trips=lambda x: 1, *rows.element_type._fields)
    | beam.GroupBy('passenger_count').aggregate_field(
        'num_trips', sum, 'total_trips').aggregate_field(
            'total_amount', mean, 'avg_fare').aggregate_field(
                'trip_distance', mean, 'avg_distance'))

    result | beam.Map(print)
    
    final_for_write = result | "ToFormalRow" >> beam.Map(to_formal_row)
    
    final_for_write | "WriteToCustomIceberg" >> beam.managed.Write(
        "iceberg", config=sink_config
    )

View the table in the [Lakehouse console](console.cloud.google.com/biglake/metastore/catalogs).

# Cleaning Up

To clean up all Google Cloud resources used in this project, you can [delete the Google Cloud project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects) you used for the tutorial.

Otherwise, you can delete the individual resources you created in this tutorial by uncommenting below:

In [ ]:
# Delete Lakehouse table and bucket
commands = [
    ["gcloud", "biglake", "iceberg", "tables", "delete", "nyc_taxicab", 
     f"--catalog={CATALOG_NAME}", "--namespace=my_namespace"],
    ["gcloud", "biglake", "iceberg", "namespaces", "delete", "my_namespace",
     f"catalog={CATALOG_NAME}"],
    ["gcloud", "biglake", "iceberg", "catalogs", "delete", CATALOG_NAME],
    ["gcloud", "storage", "rm", "--recurvise", WAREHOUSE_BUCKET_NAME]
]

for command in commands:
    result = subprocess.run(command, capture_output=True, text=True)
    
    print(result.stdout)
    print(result.stderr)